In [1]:
from azure.ai.evaluation import GroundednessEvaluator
import os
from dotenv import load_dotenv

load_dotenv()

Endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
print("AZURE OPENAI DEPLOYMENT NAME:", deployment)
print("Endpoint configured:", bool(Endpoint))   #Using bool gives a True/False instead of displaying the value.

AZURE OPENAI DEPLOYMENT NAME: gpt-4o
Endpoint configured: True


In [2]:
import azure.identity

credential = azure.identity.DefaultAzureCredential()
token_provider = azure.identity.get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

token = token_provider()

In [4]:
model_config = {
    "azure_endpoint" : os.environ.get("AZURE_OPENAI_ENDPOINT"),
    "api_key" : os.environ.get("AZURE_OPENAI_API_KEY"),
    "azure_deployment" : os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    "api_version" : os.getenv("AZURE_OPENAI_API_VERSION")
}

print("Model:", model_config["azure_deployment"])
print("Api Key:", bool(model_config["api_key"]))
print("Endpoint:", bool(model_config["azure_endpoint"]))

Model: gpt-4o
Api Key: True
Endpoint: True


In [5]:
context = """The store allows customers to return non-defective products
within 30 days of purchase, provided the products are unused,
unopened, and in their original packaging.

Proof of purchase is required for all returns.

Defective products can be returned within 60 days of purchase,
even if the packaging has been opened.

Approved refunds are processed to the original payment method
within 5 to 7 business days."""


query = """I purchased a pair of headphones 18 days ago.
I already opened the packaging and used them.
The headphones work perfectly, but I don't like the sound quality.

Can I return them and receive a refund?"""


ground_truth = """The customer is not eligible for a standard return because
the headphones have been opened and used.

Although the purchase was made within 30 days, the return
policy requires non-defective products to be unused and
unopened.

Since the headphones are functioning correctly, the
defective-product return policy does not apply."""


response = """Yes! Since you purchased your headphones only 18 days ago,
you are eligible for a return under our 30-day return policy.

You can return the headphones even though they have been
opened and used, as long as they are still working properly.

Once you return the headphones, your refund will be
processed within 24 hours.

Thank you for shopping with us!"""

In [6]:
groundedness_eval = GroundednessEvaluator(model_config)

groundedness_score = groundedness_eval(
    response=response,
    context=context,
)

print(groundedness_score)

{'groundedness': 2.0, 'groundedness_score': 2.0, 'groundedness_passed': False, 'groundedness_result': 'fail', 'groundedness_reason': "Let's think step by step: The context specifies that non-defective products can be returned within 30 days if they are unused, unopened, and in their original packaging, with proof of purchase required. Defective products can be returned within 60 days even if the packaging has been opened. Refunds are processed within 5 to 7 business days. The response incorrectly states that the headphones can be returned even though they have been opened and used, which contradicts the context's requirement for non-defective products to be unused and unopened. Additionally, the response claims that refunds will be processed within 24 hours, which is inconsistent with the context's stated timeline of 5 to 7 business days. While the response attempts to address the context, it includes incorrect information, making it unreliable.", 'groundedness_status': 'completed', 'g

In [9]:
from azure.ai.evaluation import RelevanceEvaluator

relevance_eval = RelevanceEvaluator(model_config)

relevance_score = relevance_eval(
    response=response,
    context=context,
    query=query
)

print(relevance_score)

{'relevance': 5.0, 'relevance_score': 5.0, 'relevance_passed': True, 'relevance_result': 'pass', 'relevance_reason': "The response directly addresses the user's query about returning the headphones and provides clear, accurate information about the return policy, including the eligibility timeframe, condition requirements, and refund processing time. It fully satisfies the user's request.", 'relevance_status': 'completed', 'relevance_threshold': 3, 'relevance_properties': {'prompt_tokens': 1847, 'completion_tokens': 63, 'total_tokens': 1910, 'finish_reason': 'stop', 'model': 'gpt-4o-2024-11-20', 'sample_input': '[{"role": "user", "content": "{\\"query\\": \\"I purchased a pair of headphones 18 days ago.\\\\nI already opened the packaging and used them.\\\\nThe headphones work perfectly, but I don\'t like the sound quality.\\\\n\\\\nCan I return them and receive a refund?\\", \\"response\\": \\"Yes! Since you purchased your headphones only 18 days ago,\\\\nyou are eligible for a return 

In [10]:
from azure.ai.evaluation import CoherenceEvaluator

coherence_eval = CoherenceEvaluator(model_config)

coherence_score = coherence_eval(
    response=response,
    query=query
)

print(coherence_score)

{'coherence': 4.0, 'coherence_score': 4.0, 'coherence_passed': True, 'coherence_result': 'pass', 'coherence_reason': "Let's think step by step: The RESPONSE directly addresses the QUERY by confirming the return eligibility based on the 30-day return policy. It logically explains that the headphones can be returned despite being opened and used, as long as they are still functional. The refund process is also clearly outlined, and the RESPONSE flows smoothly with appropriate transitions between ideas. The information is presented in a logical and orderly manner, making it easy to follow and understand.", 'coherence_status': 'completed', 'coherence_threshold': 3, 'coherence_properties': {'prompt_tokens': 1360, 'completion_tokens': 108, 'total_tokens': 1468, 'finish_reason': 'stop', 'model': 'gpt-4o-2024-11-20', 'sample_input': '[{"role": "user", "content": "{\\"query\\": \\"I purchased a pair of headphones 18 days ago.\\\\nI already opened the packaging and used them.\\\\nThe headphones 

In [11]:
from azure.ai.evaluation import FluencyEvaluator

fluency_eval = FluencyEvaluator(model_config)

fluency_score = fluency_eval(
    response=response,
    query=query
)

print(fluency_score)

{'fluency': 4.0, 'fluency_score': 4.0, 'fluency_passed': True, 'fluency_result': 'pass', 'fluency_reason': "Let's think step by step: The RESPONSE is clear, coherent, and grammatically accurate. It uses appropriate vocabulary and sentence structures to convey the message effectively. The sentences are well-constructed and flow logically, with no noticeable errors. While the vocabulary and sentence complexity are good, they are not exceptionally sophisticated or nuanced, which places the RESPONSE at a proficient level rather than exceptional. Overall, the RESPONSE demonstrates a high level of fluency and readability.", 'fluency_status': 'completed', 'fluency_threshold': 3, 'fluency_properties': {'prompt_tokens': 978, 'completion_tokens': 110, 'total_tokens': 1088, 'finish_reason': 'stop', 'model': 'gpt-4o-2024-11-20', 'sample_input': '[{"role": "user", "content": "{\\"response\\": \\"Yes! Since you purchased your headphones only 18 days ago,\\\\nyou are eligible for a return under our 3

In [12]:
from azure.ai.evaluation import SimilarityEvaluator

similarity_eval = SimilarityEvaluator(model_config)

similarity_score = similarity_eval(
    response=response,
    query=query,
    ground_truth=ground_truth
)

print(similarity_score)

{'similarity': 1.0, 'similarity_score': 1.0, 'similarity_passed': False, 'similarity_result': 'fail', 'similarity_reason': 'The predicted answer is not at all similar to the correct answer as it incorrectly states the return policy.', 'similarity_status': 'completed', 'similarity_threshold': 3, 'similarity_properties': {'prompt_tokens': 1121, 'completion_tokens': 41, 'total_tokens': 1162, 'finish_reason': 'stop', 'model': 'gpt-4o-2024-11-20', 'sample_input': '[{"role": "user", "content": "{\\"query\\": \\"I purchased a pair of headphones 18 days ago.\\\\nI already opened the packaging and used them.\\\\nThe headphones work perfectly, but I don\'t like the sound quality.\\\\n\\\\nCan I return them and receive a refund?\\", \\"response\\": \\"Yes! Since you purchased your headphones only 18 days ago,\\\\nyou are eligible for a return under our 30-day return policy.\\\\n\\\\nYou can return the headphones even though they have been\\\\nopened and used, as long as they are still working pro

In [13]:
from azure.ai.evaluation import F1ScoreEvaluator

f1_eval = F1ScoreEvaluator()

f1_score = f1_eval(
    response=response,
    ground_truth=ground_truth
)

print(f1_score)

{'f1_score': 0.36000000000000004, 'f1_score_score': 0.36000000000000004, 'f1_score_passed': False, 'f1_score_result': 'fail', 'f1_score_reason': None, 'f1_score_status': 'completed', 'f1_score_threshold': 0.5, 'f1_score_properties': None}


In [15]:
from azure.ai.evaluation import RougeScoreEvaluator, RougeType

rouge_eval = RougeScoreEvaluator(rouge_type=RougeType.ROUGE_1)

rouge_score = rouge_eval(
    response=response,
    ground_truth=ground_truth,
)

print(rouge_score)

{'rouge': 0.3893805309734513, 'rouge_score': 0.3893805309734513, 'rouge_passed': False, 'rouge_result': 'fail', 'rouge_reason': None, 'rouge_status': 'completed', 'rouge_threshold': 0.5, 'rouge_properties': {'rouge_precision': 0.3548387096774194, 'rouge_recall': 0.43137254901960786, 'rouge_f1_score': 0.3893805309734513, 'rouge_precision_result': 'fail', 'rouge_recall_result': 'fail', 'rouge_f1_score_result': 'fail', 'rouge_precision_passed': False, 'rouge_recall_passed': False, 'rouge_f1_score_passed': False, 'rouge_precision_threshold': 0.5, 'rouge_recall_threshold': 0.5, 'rouge_f1_score_threshold': 0.5}}


In [16]:
rouge_2_eval = RougeScoreEvaluator(rouge_type=RougeType.ROUGE_2)

rouge_l_eval = RougeScoreEvaluator(rouge_type=RougeType.ROUGE_L)

rouge_2_score = rouge_2_eval(
    response=response,
    ground_truth=ground_truth
)

rouge_l_score = rouge_l_eval(
    response=response,
    ground_truth=ground_truth
)

print("ROUGE-1:", rouge_score)
print("ROUGE-2:", rouge_2_score)
print("ROUGE-L:", rouge_l_score)

ROUGE-1: {'rouge': 0.3893805309734513, 'rouge_score': 0.3893805309734513, 'rouge_passed': False, 'rouge_result': 'fail', 'rouge_reason': None, 'rouge_status': 'completed', 'rouge_threshold': 0.5, 'rouge_properties': {'rouge_precision': 0.3548387096774194, 'rouge_recall': 0.43137254901960786, 'rouge_f1_score': 0.3893805309734513, 'rouge_precision_result': 'fail', 'rouge_recall_result': 'fail', 'rouge_f1_score_result': 'fail', 'rouge_precision_passed': False, 'rouge_recall_passed': False, 'rouge_f1_score_passed': False, 'rouge_precision_threshold': 0.5, 'rouge_recall_threshold': 0.5, 'rouge_f1_score_threshold': 0.5}}
ROUGE-2: {'rouge': 0.16216216216216217, 'rouge_score': 0.16216216216216217, 'rouge_passed': False, 'rouge_result': 'fail', 'rouge_reason': None, 'rouge_status': 'completed', 'rouge_threshold': 0.5, 'rouge_properties': {'rouge_precision': 0.14754098360655737, 'rouge_recall': 0.18, 'rouge_f1_score': 0.16216216216216217, 'rouge_precision_result': 'fail', 'rouge_recall_result': 

In [17]:
from azure.ai.evaluation import BleuScoreEvaluator

bleu_eval = BleuScoreEvaluator()

bleu_score = bleu_eval(
    response=response,
    ground_truth=ground_truth
)

print(bleu_score)

{'bleu': 0.0974613169886531, 'bleu_score': 0.0974613169886531, 'bleu_passed': False, 'bleu_result': 'fail', 'bleu_reason': None, 'bleu_status': 'completed', 'bleu_threshold': 0.5, 'bleu_properties': None}


In [18]:
from azure.ai.evaluation import MeteorScoreEvaluator

meteor_eval = MeteorScoreEvaluator(
    alpha=0.9,
    beta=3.0,
    gamma=0.5
)

meteor_score = meteor_eval(
    response=response,
    ground_truth=ground_truth,
)

print(meteor_score)

{'meteor': 0.3025372311086596, 'meteor_score': 0.3025372311086596, 'meteor_passed': False, 'meteor_result': 'fail', 'meteor_reason': None, 'meteor_status': 'completed', 'meteor_threshold': 0.5, 'meteor_properties': None}


In [7]:
from azure.ai.evaluation import GleuScoreEvaluator

gleu_eval = GleuScoreEvaluator()

gleu_score = gleu_eval(
    response=response,
    ground_truth=ground_truth,
)

print(gleu_score)

{'gleu': 0.15185185185185185, 'gleu_score': 0.15185185185185185, 'gleu_passed': False, 'gleu_result': 'fail', 'gleu_reason': None, 'gleu_status': 'completed', 'gleu_threshold': 0.5, 'gleu_properties': None}
